# 4주차 개념 사전 · CNN 구조와 효율화 블록 5가지

이번 노트북에서는 앞으로 UCI HAR 실험에서 조합할 다섯 항목을 먼저 분리해서 이해합니다.

1. ResNet
2. SENet
3. Inception
4. Depthwise Separable Convolution
5. CBAM

각 항목은 다음 순서로 확인합니다.

```text
어떤 문제를 해결하는가
→ 입력을 어떻게 바꾸는가
→ shape은 어떻게 변하는가
→ 필터링과 재가중 중 무엇인가
→ 어느 위치에 넣는가
→ UCI HAR 1D 신호에서는 어떻게 사용하는가
→ 장점과 한계는 무엇인가
```

이 파일은 실험 결과를 내는 노트북이 아니라 B1·B2 이후 모델을 설계할 때 다시 보는 **개념 참고서**입니다.

## 0. 다섯 항목은 같은 종류가 아닙니다

| 항목 | 종류 | 핵심 질문 | 대표 연산 |
|---|---|---|---|
| ResNet | 네트워크 연결 방식 | 깊은 모델을 어떻게 안정적으로 학습할까? | `Add` shortcut |
| SENet | 채널 attention 모듈 | 어떤 특징 채널이 중요한가? | 채널별 `Multiply` |
| Inception | 다중 경로 특징 추출 구조 | 어느 시간 크기의 특징이 중요한가? | 병렬 Conv + `Concatenate` |
| Depthwise Separable Conv | 효율적인 합성곱 연산 | 합성곱 계산량을 어떻게 줄일까? | depthwise + pointwise |
| CBAM | 채널·위치 attention 모듈 | 어떤 채널과 어느 위치가 중요한가? | 채널 attention + 위치 attention |

따라서 다섯 항목 중 하나만 고르는 문제가 아닙니다. 예를 들어 다음 조합이 가능합니다.

```text
Inception으로 여러 시간 범위의 특징 생성
→ SENet 또는 CBAM으로 특징 재가중
→ ResNet shortcut으로 입력과 Add
→ 일부 Conv를 depthwise separable convolution으로 경량화
```

다만 처음부터 모두 넣으면 어떤 요소가 효과를 냈는지 알 수 없으므로 하나씩 추가하며 비교해야 합니다.

## 0-1. 공통 표기: 이미지 2D와 센서 시계열 1D

원 논문 대부분은 이미지를 대상으로 하지만 UCI HAR는 시계열입니다.

| 데이터 | 일반적인 shape | 위치축 | 채널축 |
|---|---|---|---|
| 이미지 | `(batch, height, width, channels)` | 높이·너비 | 특징맵 |
| UCI HAR | `(batch, time, channels)` | 128개 시점 | 9개 센서 또는 학습 특징 |

이 노트북에서는 다음 기호를 사용합니다.

- $B$: batch 크기
- $T$: 시간 길이, UCI HAR에서는 128
- $C$: 특징 채널 수
- $K$: Conv1D kernel 크기

입력 센서가 `(B, 128, 9)`라도 첫 Conv1D를 지나면 `(B, 128, 32)`처럼 됩니다. 이때 32개 채널은 원본 센서축이 아니라 네트워크가 학습한 특징입니다.

## 0-2. 필터링과 재가중의 차이

이 차이를 먼저 알아야 SENet과 CBAM을 “새로운 필터”로 잘못 이해하지 않습니다.

### 합성곱 필터링

Conv1D는 주변 시점과 입력 채널을 섞어 **새로운 시간 특징**을 만듭니다.

$$u_{t,c}=\sum_{\tau}\sum_k w_{\tau,k,c}x_{t+\tau,k}$$

### attention 재가중

SENet과 CBAM은 이미 생성된 특징에 중요도를 곱합니다.

$$\widetilde U_{t,c}=a_{t,c}U_{t,c}$$

- SENet: 채널마다 하나의 값 $a_c$
- CBAM: 채널 중요도 $a_c$와 시간 위치 중요도 $a_t$를 차례대로 적용

따라서 합성곱은 특징을 **만들고**, attention은 만들어진 특징의 영향력을 **조절**합니다.

# 1. ResNet

## 1-1. 어떤 문제를 해결하는가?

네트워크를 깊게 만들면 더 복잡한 특징을 학습할 가능성이 있지만, 층을 추가한다고 항상 학습 오차가 줄어드는 것은 아닙니다. 매우 깊은 모델에서는 최적화가 어려워지고 더 깊은 모델의 학습 오차가 오히려 커지는 degradation 현상이 나타날 수 있습니다.

ResNet은 몇 개 층을 건너뛰는 shortcut을 만들어 깊은 모델이 잔차 함수를 학습하도록 합니다.

- 일반 블록: 목표 함수 $H(x)$를 직접 학습
- 잔차 블록: $F(x)=H(x)-x$를 학습

$$y=F(x)+x$$

필요한 변환이 거의 없다면 복잡한 항등함수를 새로 만들기보다 $F(x)\approx0$으로 만드는 편이 쉬울 수 있습니다.

## 1-2. gradient 전달과 정확한 해석

$$\frac{\partial y}{\partial x}=\frac{\partial F(x)}{\partial x}+I$$

shortcut의 항등 성분 $I$가 정보와 gradient가 전달될 직접 경로를 제공합니다. 하지만 다음과 같이 해석해야 합니다.

- gradient 소실이 절대로 발생하지 않는다는 보장이 아닙니다.
- residual connection은 깊은 모델의 최적화를 쉽게 만드는 장치입니다.
- 얕은 모델에서는 효과가 작을 수 있습니다.
- 작은 데이터에서는 깊이가 늘면서 과적합이 더 커질 수 있습니다.
- 학습 안정성과 최종 정확도 향상은 별도로 확인해야 합니다.

## 1-3. `Add`와 shape

`Add`는 같은 위치의 값을 더하므로 두 입력의 shape이 같아야 합니다.

```text
shortcut : (B, 128, 32)
residual : (B, 128, 32)
Add 결과 : (B, 128, 32)
```

채널이나 시간 길이가 다르면 shortcut에 1×1 Conv1D와 stride를 적용합니다.

```text
입력             : (B, 128, 32)
residual 출력     : (B, 64, 64)
projection shortcut: (B, 64, 64)
Add 결과          : (B, 64, 64)
```

## 1-4. UCI HAR용 1D residual block

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model


def residual_block_1d(x, filters, stride=1, name="res"):
    shortcut = x

    y = layers.Conv1D(
        filters, 3, strides=stride, padding="same",
        use_bias=False, name=f"{name}_conv1",
    )(x)
    y = layers.BatchNormalization(name=f"{name}_bn1")(y)
    y = layers.Activation("relu", name=f"{name}_relu1")(y)

    y = layers.Conv1D(
        filters, 3, padding="same",
        use_bias=False, name=f"{name}_conv2",
    )(y)
    y = layers.BatchNormalization(name=f"{name}_bn2")(y)

    # Add 전에 시간 길이와 채널 수를 맞춥니다.
    if stride != 1 or x.shape[-1] != filters:
        shortcut = layers.Conv1D(
            filters, 1, strides=stride, padding="same",
            use_bias=False, name=f"{name}_projection",
        )(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_projection_bn")(shortcut)

    y = layers.Add(name=f"{name}_add")([shortcut, y])
    return layers.Activation("relu", name=f"{name}_out")(y)

## 1-5. 장점과 한계

장점:

- 깊은 모델의 최적화를 돕습니다.
- shape이 같으면 shortcut에 추가 파라미터가 없습니다.
- 반복 가능한 블록으로 구조가 규칙적입니다.
- Inception, SE, CBAM, depthwise convolution과 결합할 수 있습니다.

한계:

- 깊이가 늘면 계산량과 메모리도 증가합니다.
- shape이 다르면 projection shortcut이 필요합니다.
- 작은 데이터에서는 과적합할 수 있습니다.
- shortcut이 있다고 항상 더 높은 정확도가 나오는 것은 아닙니다.

# 2. SENet

## 2-1. 어떤 문제를 해결하는가?

CNN이 만든 모든 특징 채널이 현재 입력에서 똑같이 중요하지는 않습니다. SENet의 SE 블록은 전체 위치 정보를 요약해 특징 채널 사이의 관계를 학습하고, 중요한 채널은 상대적으로 유지하며 덜 유용한 채널은 억제합니다.

입력 특징을 $U\in\mathbb{R}^{T\times C}$라고 하겠습니다.

```text
U (B, T, C)
→ Squeeze  (B, C)
→ Excitation (B, C/r) → (B, C)
→ Reshape (B, 1, C)
→ U와 Multiply
→ 출력 (B, T, C)
```

## 2-2. Squeeze · Excitation · Scale

### Squeeze

각 채널을 시간축 전체에서 평균냅니다.

$$z_c=\frac{1}{T}\sum_{t=1}^{T}U_{t,c}$$

### Excitation

두 Dense 층으로 채널 관계를 학습합니다.

$$s=\sigma(W_2\operatorname{ReLU}(W_1z))$$

중간 크기는 보통 $C/r$이며 $r$을 reduction ratio라고 합니다.

### Scale

$$\widetilde U_{t,c}=s_cU_{t,c}$$

중요한 점은 $U$ 자체를 $C/r$ 채널로 영구 압축하는 것이 아니라, **가중치를 계산하는 작은 제어 경로만 압축**한다는 것입니다. 최종 출력 shape은 입력과 같습니다.

## 2-3. ResNet과 결합할 때 위치

표준적인 SE-ResNet은 residual 변환 뒤, shortcut과 더하기 전에 SE를 둡니다.

```text
x ────────────────────────────┐
                              Add → 출력
x → residual Conv → SE ───────┘
```

$$y=x+SE(F(x))\odot F(x)$$

이렇게 하면 identity shortcut은 그대로 보존되고 residual branch가 만든 특징만 재가중됩니다. `Add` 뒤에 SE를 두는 변형도 가능하지만 shortcut까지 함께 조절하므로 다른 가설을 검증하는 모델이 됩니다.

## 2-4. UCI HAR용 1D SE block

In [ ]:
def se_block_1d(x, reduction_ratio=8, name="se"):
    channels = int(x.shape[-1])
    hidden = max(channels // reduction_ratio, 1)

    # 시간 정보를 채널별 하나의 값으로 요약합니다.
    scale = layers.GlobalAveragePooling1D(name=f"{name}_squeeze")(x)

    # 채널 중요도를 계산하는 제어 경로입니다.
    scale = layers.Dense(hidden, activation="relu", name=f"{name}_reduce")(scale)
    scale = layers.Dense(channels, activation="sigmoid", name=f"{name}_expand")(scale)
    scale = layers.Reshape((1, channels), name=f"{name}_reshape")(scale)

    return layers.Multiply(name=f"{name}_scale")([x, scale])

## 2-5. 파라미터와 한계

bias를 제외하면 SE의 주요 추가 파라미터는 다음과 같습니다.

$$\frac{C^2}{r}+\frac{C^2}{r}=\frac{2C^2}{r}$$

장점:

- 기존 CNN 블록에 쉽게 추가할 수 있습니다.
- 시간축 전체 정보를 이용해 채널 관계를 학습합니다.
- 원래 feature map shape을 유지합니다.

한계:

- 어느 시간 위치가 중요한지는 직접 구분하지 않습니다.
- 원본 9개 센서축이 아니라 학습 특징 채널을 재가중합니다.
- Global Average Pooling이 시간 위치 정보를 압축합니다.
- reduction ratio가 너무 크면 작은 모델에서 제어 경로가 지나치게 좁아집니다.

# 3. Inception

## 3-1. 어떤 문제를 해결하는가?

중요한 패턴의 크기를 하나의 kernel로 미리 정하기 어렵습니다. Inception은 같은 입력에 여러 연산을 병렬로 적용하여 한 블록에서 여러 수용 영역의 특징을 만듭니다.

UCI HAR 예시는 다음과 같습니다.

```text
입력 (B, 128, 32)
 ├─ Conv1D kernel 3  → (B, 128, 8)
 ├─ Conv1D kernel 7  → (B, 128, 8)
 ├─ Conv1D kernel 15 → (B, 128, 8)
 └─ MaxPool + Conv1D → (B, 128, 8)
             ↓ Concatenate
출력 (B, 128, 32)
```

UCI HAR가 50Hz이므로 kernel 3·7·15는 한 블록에서 대략 0.06초·0.14초·0.30초의 국소 범위를 봅니다. 블록을 여러 개 쌓으면 유효 수용 영역은 더 넓어집니다.

## 3-2. Bottleneck 1×1 convolution

큰 kernel을 모든 입력 채널에 바로 적용하면 계산량이 커집니다. 먼저 kernel 1의 Conv1D로 채널을 줄인 뒤 큰 kernel을 적용할 수 있습니다.

일반 Conv1D의 파라미터 수는 bias를 제외하면 다음과 같습니다.

$$K C_{in}C_{out}$$

예를 들어 입력 64채널에 kernel 15, 출력 16채널을 직접 적용하면:

$$15\times64\times16=15{,}360$$

먼저 1×1 Conv1D로 64→16채널로 줄이면:

$$1\times64\times16 + 15\times16\times16=4{,}864$$

병목은 계산량을 줄이지만 너무 좁으면 정보가 손실될 수 있으므로 중간 채널 수도 실험 대상입니다.

## 3-3. `Concatenate`와 `Add`는 다릅니다

### Inception의 Concatenate

채널을 이어 붙입니다.

```text
(B,T,8) + (B,T,8) + (B,T,8) + (B,T,8)
→ (B,T,32)
```

### ResNet의 Add

같은 위치의 값을 더하며 shape이 유지됩니다.

```text
(B,T,32) + (B,T,32)
→ (B,T,32)
```

Inception-ResNet에서는 Inception 내부 branch를 `Concatenate`한 뒤 projection으로 채널을 맞추고 shortcut과 `Add`합니다.

## 3-4. UCI HAR용 1D Inception block

In [ ]:
def inception_block_1d(
    x,
    filters=32,
    bottleneck_filters=16,
    kernel_sizes=(3, 7, 15),
    name="inc",
):
    if filters % 4 != 0:
        raise ValueError("filters는 4의 배수여야 합니다.")

    branch_filters = filters // 4
    bottleneck = layers.Conv1D(
        bottleneck_filters, 1, padding="same",
        activation="relu", name=f"{name}_bottleneck",
    )(x)

    branches = [
        layers.Conv1D(
            branch_filters, kernel, padding="same",
            activation="relu", name=f"{name}_k{kernel}",
        )(bottleneck)
        for kernel in kernel_sizes
    ]

    pool = layers.MaxPooling1D(3, strides=1, padding="same", name=f"{name}_pool")(x)
    pool = layers.Conv1D(
        branch_filters, 1, padding="same",
        activation="relu", name=f"{name}_pool_projection",
    )(pool)
    branches.append(pool)

    return layers.Concatenate(axis=-1, name=f"{name}_concat")(branches)

## 3-5. 장점과 한계

장점:

- 여러 시간 범위의 특징을 동시에 추출합니다.
- bottleneck으로 큰 kernel의 비용을 제한할 수 있습니다.
- 동적 행동처럼 서로 다른 주기의 패턴에 자연스러운 가설입니다.

한계:

- 분기와 채널 수 설계가 복잡합니다.
- 이론적 연산량 감소가 실제 장치 속도와 항상 일치하지 않습니다.
- SIT/STAND처럼 중력 방향이 중요한 문제를 자동으로 해결하지 않습니다.
- kernel 3·7·15가 UCI HAR의 정답이라는 뜻은 아닙니다.

# 4. Depthwise Separable Convolution

## 4-1. 어떤 문제를 해결하는가?

일반 합성곱은 시간 패턴 추출과 채널 결합을 한 번에 합니다. Depthwise separable convolution은 이를 두 단계로 나눕니다.

1. **Depthwise convolution:** 각 입력 채널에 별도의 시간 필터 적용
2. **Pointwise convolution:** kernel 1로 채널 정보 결합

```text
입력 (B,T,Cin)
→ Depthwise: 채널별 시간 필터
→ Pointwise: 채널 혼합
→ 출력 (B,T,Cout)
```

Depthwise만 사용하면 서로 다른 채널을 충분히 섞지 못하므로 일반적으로 pointwise와 함께 사용합니다.

## 4-2. 파라미터 수 비교

### 일반 Conv1D

$$K C_{in}C_{out}$$

### Depthwise separable Conv1D

$$K C_{in}+C_{in}C_{out}$$

예를 들어 $K=7$, $C_{in}=32$, $C_{out}=64$라면:

- 일반 Conv1D: $7\times32\times64=14{,}336$
- depthwise: $7\times32=224$
- pointwise: $32\times64=2{,}048$
- 합계: $2{,}272$

이 예에서는 파라미터가 약 84.2% 줄어듭니다. 그러나 실제 속도는 하드웨어의 연산 최적화와 메모리 접근에도 영향을 받습니다.

## 4-3. Keras 1D 코드

In [ ]:
def separable_conv_block_1d(x, filters, kernel_size=7, name="sep"):
    # Keras가 depthwise와 pointwise 단계를 묶어서 제공합니다.
    x = layers.SeparableConv1D(
        filters,
        kernel_size,
        padding="same",
        use_bias=False,
        name=f"{name}_conv",
    )(x)
    x = layers.BatchNormalization(name=f"{name}_bn")(x)
    return layers.Activation("relu", name=f"{name}_out")(x)

## 4-4. Inception·ResNet과의 관계

Depthwise separable convolution은 독립적인 attention 모듈이 아니라 Conv를 대체하는 연산입니다.

```text
기존 Inception branch: Conv1D(kernel 15)
경량 branch          : SeparableConv1D(kernel 15)
```

```text
기존 residual branch: Conv1D → Conv1D
경량 residual branch: SeparableConv1D → SeparableConv1D
```

장점은 파라미터와 연산량 감소입니다. 반면 채널별 시간 필터와 채널 혼합을 분리하므로 일반 Conv보다 표현력이 낮아질 수 있습니다. 따라서 정확도뿐 아니라 파라미터, 학습 시간, 추론 시간을 함께 평가해야 합니다.

# 5. CBAM

## 5-1. 어떤 문제를 해결하는가?

SENet은 “어떤 채널이 중요한가?”만 학습합니다. CBAM은 여기에 “어느 위치가 중요한가?”를 추가합니다.

이미지에서는 channel attention 뒤 spatial attention을 적용합니다. UCI HAR 1D에서는 spatial attention을 **temporal attention**으로 바꾸어 어느 시점이 중요한지 학습합니다.

```text
입력 U
→ Channel Attention: 무엇이 중요한가?
→ Temporal Attention: 언제가 중요한가?
→ 출력
```

$$U'=M_c(U)\odot U$$

$$U''=M_t(U')\odot U'$$

## 5-2. Channel attention

CBAM channel attention은 시간축에 대해 평균 풀링과 최대 풀링을 모두 사용합니다.

```text
U (B,T,C)
 ├─ Global Average Pooling → (B,C)
 └─ Global Max Pooling     → (B,C)
          ↓ 같은 MLP를 공유
          ↓ 두 결과 Add
          ↓ sigmoid
채널 attention (B,1,C)
```

$$M_c(U)=\sigma(MLP(AvgPool(U))+MLP(MaxPool(U)))$$

평균은 전체적인 반응을, 최댓값은 가장 강한 반응을 요약합니다. 두 경로의 MLP 가중치는 공유합니다.

## 5-3. Temporal attention

채널 attention이 적용된 특징 $U'$를 채널축으로 평균·최대 풀링합니다.

```text
U' (B,T,C)
 ├─ 채널 평균 → (B,T,1)
 └─ 채널 최대 → (B,T,1)
       ↓ Concatenate
      (B,T,2)
       ↓ Conv1D(kernel 7) + sigmoid
시간 attention (B,T,1)
```

$$M_t(U')=\sigma(f^7([AvgPool_c(U');MaxPool_c(U')]))$$

SE가 한 채널의 모든 시점에 같은 가중치를 곱하는 것과 달리 CBAM은 시점별 중요도도 조절합니다.

## 5-4. UCI HAR용 1D CBAM 코드

In [ ]:
def cbam_1d(x, reduction_ratio=8, temporal_kernel=7, name="cbam"):
    channels = int(x.shape[-1])
    hidden = max(channels // reduction_ratio, 1)

    # Channel attention의 두 pooling 경로가 같은 MLP를 공유합니다.
    shared_reduce = layers.Dense(hidden, activation="relu", name=f"{name}_shared_reduce")
    shared_expand = layers.Dense(channels, name=f"{name}_shared_expand")

    avg_descriptor = layers.GlobalAveragePooling1D(name=f"{name}_channel_avg")(x)
    max_descriptor = layers.GlobalMaxPooling1D(name=f"{name}_channel_max")(x)

    avg_score = shared_expand(shared_reduce(avg_descriptor))
    max_score = shared_expand(shared_reduce(max_descriptor))
    channel_score = layers.Add(name=f"{name}_channel_add")([avg_score, max_score])
    channel_score = layers.Activation("sigmoid", name=f"{name}_channel_sigmoid")(channel_score)
    channel_score = layers.Reshape((1, channels), name=f"{name}_channel_reshape")(channel_score)
    x = layers.Multiply(name=f"{name}_channel_scale")([x, channel_score])

    # Temporal attention: 각 시점에서 채널 평균과 최댓값을 사용합니다.
    temporal_avg = layers.Lambda(
        lambda tensor: tf.reduce_mean(tensor, axis=-1, keepdims=True),
        name=f"{name}_temporal_avg",
    )(x)
    temporal_max = layers.Lambda(
        lambda tensor: tf.reduce_max(tensor, axis=-1, keepdims=True),
        name=f"{name}_temporal_max",
    )(x)
    temporal_descriptor = layers.Concatenate(
        axis=-1, name=f"{name}_temporal_concat"
    )([temporal_avg, temporal_max])
    temporal_score = layers.Conv1D(
        1,
        temporal_kernel,
        padding="same",
        activation="sigmoid",
        use_bias=False,
        name=f"{name}_temporal_conv",
    )(temporal_descriptor)

    return layers.Multiply(name=f"{name}_temporal_scale")([x, temporal_score])

## 5-5. SENet과 CBAM 비교

| 구분 | SENet | CBAM |
|---|---|---|
| 질문 | 어떤 채널이 중요한가? | 어떤 채널과 어느 시점이 중요한가? |
| pooling | 시간 평균 | 시간 평균+최대, 채널 평균+최대 |
| 출력 attention | `(B,1,C)` | `(B,1,C)` 후 `(B,T,1)` |
| 시간 위치 구분 | 하지 않음 | 수행함 |
| 구조와 비용 | 더 단순 | 조금 더 복잡 |
| UCI HAR 가설 | 전역적인 특징 선택 | 특징 선택 + 중요한 동작 시점 강조 |

CBAM이 항상 SENet보다 좋은 것은 아닙니다. 정적 자세처럼 윈도우 전체가 비슷한 경우에는 temporal attention의 이점이 작을 수 있고, 작은 데이터에서는 추가 자유도가 과적합으로 이어질 수 있습니다.

## 5-6. CBAM의 순서도 실험 대상입니다

원 논문의 기본 순서는 channel attention 다음 spatial attention입니다.

```text
Channel → Temporal
```

순서를 바꾸면 같은 계산이 아닙니다.

```text
Temporal → Channel
Parallel 후 결합
Channel만 사용
Temporal만 사용
```

앞 단계가 특징을 바꾸면 뒤 단계가 보는 입력도 달라지므로 attention 연산은 일반적으로 교환되지 않습니다. 첫 실험은 원 논문의 순서를 1D로 옮긴 `Channel → Temporal`을 기준으로 삼는 것이 해석하기 쉽습니다.

# 6. 다섯 항목을 한눈에 비교하기

| 항목 | 주된 목적 | 특징 생성 | 채널 재가중 | 시간 위치 재가중 | 깊은 학습 보조 | 연산 절감 |
|---|---|---:|---:|---:|---:|---:|
| ResNet | 최적화와 정보 전달 | 간접적 | 아니오 | 아니오 | 예 | 아니오 |
| SENet | 채널 선택 | 아니오 | 예 | 아니오 | 아니오 | 아니오 |
| Inception | 다중 시간 범위 특징 | 예 | 아니오 | 아니오 | 아니오 | bottleneck으로 일부 |
| Depthwise separable conv | 합성곱 경량화 | 예 | 아니오 | 아니오 | 아니오 | 예 |
| CBAM | 채널·위치 선택 | 아니오 | 예 | 예 | 아니오 | 아니오 |

한 문장으로 정리하면 다음과 같습니다.

- ResNet: 깊어져도 학습하기 쉽도록 **길을 추가**합니다.
- SENet: 만들어진 특징 중 **무엇이 중요한지** 조절합니다.
- Inception: 서로 다른 시간 크기의 특징을 **동시에 만듭니다**.
- Depthwise separable convolution: 특징 생성 과정을 **저비용으로 분리**합니다.
- CBAM: **무엇과 언제가 중요한지** 차례대로 조절합니다.

# 7. 조합할 때 연산 순서

## 7-1. 현재 B2: Inception → SE → Residual Add

```text
x ───────────────────────────────────┐
                                     Add → ReLU
x → Inception → projection → SE ─────┘
```

- Inception이 특징을 만듭니다.
- SE가 생성된 residual 특징 채널을 재가중합니다.
- shortcut은 그대로 보존합니다.

## 7-2. 다음 후보: Inception → CBAM → Residual Add

```text
x ─────────────────────────────────────┐
                                       Add → ReLU
x → Inception → projection → CBAM ─────┘
```

SE 대신 CBAM을 넣어 채널뿐 아니라 중요한 시점도 조절하는 가설입니다.

## 7-3. 경량 후보: Depthwise Inception → attention → Add

Inception의 큰 Conv1D를 SeparableConv1D로 교체하여 계산량을 줄입니다. 이때 목적은 최고 정확도만이 아니라 정확도 대비 파라미터와 추론 시간입니다.

# 8. 순서는 관계가 없을까?

순서는 중요합니다. 합성곱 $F$와 attention $A$는 일반적으로 교환되지 않습니다.

$$F(A(x)\odot x)\neq A(F(x))\odot F(x)$$

| 위치 | 무엇을 조절하는가? | 검증하는 질문 |
|---|---|---|
| Inception 이전 | 입력 특징 | 필터에 들어가기 전 특징 선택이 유리한가? |
| Inception 각 branch | 시간 범위별 특징 | branch별 선택이 유리한가? |
| Inception 이후·Add 이전 | 생성된 residual 특징 | 최종 residual 특징 재가중이 유리한가? |
| Add 이후 | shortcut+residual 전체 | 합쳐진 특징 전체 재가중이 유리한가? |

첫 실험에서는 원인을 분리하기 쉬운 위치 하나를 고정하고, 효과가 확인된 뒤 위치 ablation을 하는 것이 좋습니다.

# 9. UCI HAR 실험에서 확인할 공통 기준

구조 이름만 바꾸고 조건이 달라지면 공정한 비교가 아닙니다.

고정할 조건:

- 동일한 피험자 단위 train/validation 분리
- 학습 데이터로만 계산한 정규화
- 동일한 seed 0·1·2
- 동일 optimizer, learning rate, batch size, 최대 epoch
- 동일 EarlyStopping 기준
- 최종 모델 선택 전 공식 test 미사용

기록할 성능:

- Validation Accuracy
- Macro F1
- 클래스별 recall
- 피험자별 정확도
- SIT/STAND 및 동적 행동 오류

기록할 효율성:

- 파라미터 수
- best epoch
- 학습 시간
- 샘플당 추론 시간

세 seed의 평균·표준편차와 같은 seed에서의 차이를 함께 확인합니다.

# 10. 교수님 권장 방식: 단독 효과와 조합 효과를 분리하기

## 10-1. 1단계 · 각 항목의 단독 효과

모든 실험은 같은 입력, 분할, 학습 조건과 비슷한 규모의 분류 head를 사용합니다.

| 실험 | 구성 | 단독 효과의 의미 |
|---|---|---|
| A0 | 기준 CNN | 모든 비교의 기준 |
| A1 | ResNet 1D | 일반 CNN 대비 shortcut과 residual block의 효과 |
| A2 | Inception 1D | 단일 kernel 대비 병렬 멀티스케일 특징의 효과 |
| A3 | 기준 CNN + SENet | 기준 CNN에 채널 attention만 추가한 효과 |
| A4 | Depthwise CNN | 기준 CNN의 일반 Conv만 separable Conv로 교체한 효과 |
| A5 | 기준 CNN + CBAM | 기준 CNN에 채널+시간 attention만 추가한 효과 |

여기서 SENet·CBAM은 그 자체로 입력부터 분류까지 수행하는 독립 backbone이 아닙니다. 따라서 “단독 SENet”은 **동일한 기준 CNN에 SE만 추가**, “단독 CBAM”은 **동일한 기준 CNN에 CBAM만 추가**했다는 뜻입니다. Depthwise도 동일하게 일반 Conv를 효율적인 연산으로 교체한 대조 실험입니다.

## 10-2. 2단계 · 별도의 조합 실험

1단계 결과와 별개로 조합 자체의 성능을 확인합니다.

| 실험 | 조합 | 확인하려는 것 |
|---|---|---|
| B1 | Inception + ResNet | 멀티스케일 특징과 residual 연결의 결합 |
| B2 | B1 + SENet | 결합 특징에 채널 attention을 더한 효과 |
| B3 후보 | B1 + CBAM | 결합 특징에 채널+시간 attention을 더한 효과 |
| B4 후보 | Depthwise Inception + ResNet | B1을 경량화했을 때의 성능·효율 trade-off |

## 10-3. 무엇을 결론 낼 수 있는가?

- `A3 - A0`: 기준 CNN에서 SE 자체가 도움이 되었는가?
- `B2 - B1`: Inception-ResNet 조합에서도 SE가 추가로 도움이 되었는가?
- 두 차이가 모두 양수: SE 효과가 비교적 일관될 가능성
- A3는 개선, B2는 하락: SE와 Inception-ResNet 사이의 상호작용 가능성
- 단독 모델은 낮지만 조합은 개선: 개별 우수성보다 조합 시너지가 있을 가능성

단독 실험과 조합 실험은 서로 대체하지 않습니다. 전자는 각 요소의 주효과를, 후자는 요소 사이의 상호작용을 보기 위한 실험입니다.

# 11. 자주 혼동하는 표현

| 부정확한 표현 | 더 정확한 표현 |
|---|---|
| Inception이 뉴런을 필터링한다 | 여러 kernel의 합성곱으로 멀티스케일 특징을 생성한다 |
| SENet이 센서축을 직접 선택한다 | 학습된 특징 채널을 입력별로 재가중한다 |
| SE의 압축으로 특징이 영구 손실된다 | attention 가중치 계산 경로만 잠시 축소했다 다시 확장한다 |
| ResNet이 gradient 소실을 없앤다 | shortcut이 정보와 gradient 전달 경로를 제공해 최적화를 돕는다 |
| Depthwise convolution만으로 채널 관계도 학습한다 | pointwise convolution이 별도로 채널을 결합한다 |
| CBAM은 새로운 특징 필터다 | 기존 특징의 채널과 위치 중요도를 재가중하는 attention이다 |
| 더 복잡하게 조합하면 반드시 좋다 | 추가 모듈의 효과는 같은 조건의 ablation으로 검증해야 한다 |

# 12. 최종 요약

```text
ResNet   = 더 깊게 학습하기 위한 shortcut
SENet    = 특징 채널별 중요도
Inception= 여러 시간 범위의 특징 생성
Depthwise separable convolution = 시간 필터링과 채널 결합을 분리한 경량 합성곱
CBAM     = 채널 중요도 + 시간 위치 중요도
```

이 다섯 방법은 서로 경쟁하는 동일한 선택지가 아니라 서로 다른 문제를 해결합니다. 따라서 좋은 조합은 가장 많은 모듈을 넣은 모델이 아니라, 데이터에서 실제로 필요한 문제를 적은 비용으로 해결하고 그 효과가 반복 실험에서 확인된 모델입니다.

# 13. 참고 자료

- [He et al., Deep Residual Learning for Image Recognition, 2015](https://arxiv.org/abs/1512.03385)
- [Hu et al., Squeeze-and-Excitation Networks, CVPR 2018](https://openaccess.thecvf.com/content_cvpr_2018/html/Hu_Squeeze-and-Excitation_Networks_CVPR_2018_paper.html)
- [Szegedy et al., Going Deeper with Convolutions, 2014](https://arxiv.org/abs/1409.4842)
- [Chollet, Xception: Deep Learning with Depthwise Separable Convolutions, 2016](https://arxiv.org/abs/1610.02357)
- [Woo et al., CBAM: Convolutional Block Attention Module, ECCV 2018](https://openaccess.thecvf.com/content_ECCV_2018/html/Sanghyun_Woo_Convolutional_Block_Attention_ECCV_2018_paper)

원 논문은 주로 2D 이미지 구조를 설명합니다. 이 노트북의 Conv1D·temporal attention 코드는 해당 개념을 UCI HAR 시계열 shape에 맞게 단순화한 수업용 구현이며, 원 논문의 전체 네트워크를 그대로 재현한 것은 아닙니다.